# Carga de las ocho tablas de Olist en MySQL

Guarda este notebook en la carpeta `notebooks/` del repositorio. Ejecuta previamente los notebooks de limpieza de los dos integrantes para generar los ocho CSV de `datos/limpios/`, y crea las tablas con `sql/create.sql` si todavía faltan.

Después selecciona tu entorno de Python y ejecuta todas las celdas. La contraseña se solicita durante la ejecución. La carga inserta registros nuevos y actualiza los registros con la misma clave primaria a partir del CSV seleccionado. Las ocho tablas se escriben en una transacción.

Al final se muestra el número de filas del CSV y el de MySQL para cada tabla. Este notebook se ha revisado con el esquema actual del repositorio. La conexión y la carga efectiva se comprueban al ejecutarlo en tu ordenador.

In [3]:
%pip install sqlalchemy pymysql

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Localizar los CSV limpios

Se reconocen tanto nombres cortos como `products.csv` como los nombres de Kaggle, por ejemplo `olist_products_dataset.csv`. Si existen los dos nombres en `datos/limpios/`, se utiliza el nombre corto. La tabla siguiente muestra el archivo elegido para cada carga.

In [4]:
from pathlib import Path
from decimal import Decimal
from getpass import getpass

import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine, inspect, text, types
from sqlalchemy.engine import URL
from sqlalchemy.dialects.mysql import insert as mysql_insert

# Si hace falta, indica la ruta completa de olist-sql-project.
project_dir_override = None
start = Path.cwd().resolve()
candidates = [Path(project_dir_override).expanduser().resolve()] if project_dir_override else [start, *start.parents]
project_dir = next(
    (folder for folder in candidates if (folder / "datos").is_dir() and (folder / "sql" / "create.sql").is_file()),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Abre este notebook dentro de olist-sql-project o indica su ruta en project_dir_override.")

clean_dir = project_dir / "datos" / "limpios"
file_options = {
    "category_translation": ("category_translation.csv", "product_category_name_translation.csv"),
    "customers": ("customers.csv", "olist_customers_dataset.csv"),
    "sellers": ("sellers.csv", "olist_sellers_dataset.csv"),
    "orders": ("orders.csv", "olist_orders_dataset.csv"),
    "products": ("products.csv", "olist_products_dataset.csv"),
    "order_reviews": ("order_reviews.csv", "olist_order_reviews_dataset.csv"),
    "order_payments": ("order_payments.csv", "olist_order_payments_dataset.csv"),
    "order_items": ("order_items.csv", "olist_order_items_dataset.csv"),
}

selected_files = {
    table: next((clean_dir / name for name in names if (clean_dir / name).is_file()), None)
    for table, names in file_options.items()
}
missing = [table for table, path in selected_files.items() if path is None]
if missing:
    raise FileNotFoundError(
        f"Faltan CSV limpios para {', '.join(missing)} en {clean_dir}. "
        "Ejecuta los notebooks de limpieza de ambos integrantes antes de cargar."
    )

tables = {
    table: pd.read_csv(path, dtype="string", keep_default_na=False, na_values=[""])
    for table, path in selected_files.items()
}
if any(frame.empty for frame in tables.values()):
    raise ValueError("Algún CSV limpio tiene cero filas. Revisa las salidas de limpieza.")

print(f"Carpeta de datos limpios: {clean_dir}")
display(pd.DataFrame([
    {"tabla": table, "archivo": selected_files[table].name, "filas_csv": len(frame)}
    for table, frame in tables.items()
]))

Carpeta de datos limpios: C:\Users\emab6\Documents\ironhack 2026\mini_proyectos\olist-sql-project\datos\limpios


,tabla,archivo,filas_csv
0,category_translation,product_category_name_translation.csv,73
1,customers,customers.csv,99441
2,sellers,olist_sellers_dataset.csv,3095
3,orders,orders.csv,99441
4,products,olist_products_dataset.csv,32951
5,order_reviews,order_reviews.csv,98673
6,order_payments,order_payments.csv,103883
7,order_items,olist_order_items_dataset.csv,112650


## 2. Conectar a MySQL

Utiliza la contraseña de la misma conexión que abres en Workbench. Los valores iniciales son usuario `root`, servidor `localhost`, puerto `3306` y base `olist`. La conexión admite contraseñas con caracteres especiales.

In [9]:
usuario = "root"
host = "localhost"
puerto = 3306
base_datos = "olist"
password = getpass("Contraseña de MySQL: ")

connection_url = URL.create(
    "mysql+pymysql",
    username=usuario,
    password=password,
    host=host,
    port=puerto,
    database=base_datos,
    query={"charset": "utf8mb4"},
)
engine = create_engine(connection_url, pool_pre_ping=True)
del password

with engine.connect() as connection:
    print(f"Conexión confirmada a la base {connection.scalar(text('SELECT DATABASE()'))}.")

Conexión confirmada a la base olist.


## 3. Revisar y cargar las ocho tablas

Se comprueban las columnas, las claves primarias y las referencias antes de comenzar las inserciones. Los códigos postales se mantienen como texto, las fechas se convierten a fechas de Python y los importes a decimales.

La carga utiliza las claves primarias para actualizar registros existentes, por lo que puedes repetirla después de una ejecución anterior. Cada CSV limpio representa los datos de su tabla. Comprueba el recuento final y cualquier diferencia entre las columnas `filas_csv` y `filas_mysql`.

In [10]:
primary_keys = {}


def insert_or_update(sql_table, connection, keys, data_iter):
    rows = [dict(zip(keys, row)) for row in data_iter]
    statement = mysql_insert(sql_table.table).values(rows)
    updates = {
        column: statement.inserted[column]
        for column in keys
        if column not in primary_keys[sql_table.name]
    }
    result = connection.execute(statement.on_duplicate_key_update(**updates))
    return result.rowcount


relationships = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_reviews", "order_id", "orders", "order_id"),
    ("order_payments", "order_id", "orders", "order_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "seller_id", "sellers", "seller_id"),
]

try:
    with engine.begin() as connection:
        schema = inspect(connection)
        available_tables = set(schema.get_table_names())
        missing_tables = set(tables).difference(available_tables)
        if missing_tables:
            raise ValueError(f"Faltan tablas en MySQL: {', '.join(sorted(missing_tables))}. Revisa sql/create.sql.")

        for table, frame in tables.items():
            columns = schema.get_columns(table)
            sql_columns = {column["name"] for column in columns}
            if set(frame.columns) != sql_columns:
                raise ValueError(
                    f"Columnas diferentes en {table}. "
                    f"Solo en CSV: {sorted(set(frame.columns) - sql_columns)}. "
                    f"Solo en MySQL: {sorted(sql_columns - set(frame.columns))}."
                )
            primary_keys[table] = schema.get_pk_constraint(table)["constrained_columns"]
            if not primary_keys[table]:
                raise ValueError(f"La tabla {table} requiere una clave primaria para identificar cada registro.")

            for column in columns:
                name = column["name"]
                sql_type = column["type"]
                if isinstance(sql_type, types.DateTime):
                    frame[name] = pd.to_datetime(frame[name], format="mixed", errors="raise")
                elif isinstance(sql_type, types.Integer):
                    frame[name] = pd.to_numeric(frame[name], errors="raise").astype("Int64")
                elif isinstance(sql_type, types.Numeric):
                    frame[name] = frame[name].map(lambda value: Decimal(value) if pd.notna(value) else None)

            keys = frame[primary_keys[table]]
            if keys.isna().any().any() or keys.duplicated().any():
                raise ValueError(f"El CSV de {table} contiene claves primarias vacías o repetidas. Revisa su limpieza.")

        for child, column, parent, parent_column in relationships:
            has_value = tables[child][column].notna()
            unmatched = has_value & ~tables[child][column].isin(tables[parent][parent_column])
            if unmatched.any():
                raise ValueError(f"{child}.{column} tiene {int(unmatched.sum())} referencias sin coincidencia en {parent}.")

        for table, frame in tables.items():
            print(f"Cargando {table}: {len(frame):,} filas...", flush=True)
            frame.to_sql(
                table,
                con=connection,
                if_exists="append",
                index=False,
                chunksize=1000,
                method=insert_or_update,
            )

        counts = pd.DataFrame([
            {
                "tabla": table,
                "filas_csv": len(frame),
                "filas_mysql": connection.scalar(text(f"SELECT COUNT(*) FROM `{table}`")),
            }
            for table, frame in tables.items()
        ])
except Exception:
    print("La carga se ha interrumpido. Revisa el mensaje de error antes de volver a ejecutarla.")
    raise
finally:
    engine.dispose()

counts["coinciden"] = counts["filas_csv"].eq(counts["filas_mysql"])
print("Carga confirmada en MySQL.")
display(counts)

Cargando category_translation: 73 filas...
Cargando customers: 99,441 filas...
Cargando sellers: 3,095 filas...
Cargando orders: 99,441 filas...
Cargando products: 32,951 filas...
Cargando order_reviews: 98,673 filas...
Cargando order_payments: 103,883 filas...
Cargando order_items: 112,650 filas...
Carga confirmada en MySQL.


,tabla,filas_csv,filas_mysql,coinciden
0,category_translation,73,73,True
1,customers,99441,99441,True
2,sellers,3095,3095,True
3,orders,99441,99441,True
4,products,32951,32951,True
5,order_reviews,98673,98673,True
6,order_payments,103883,103883,True
7,order_items,112650,112650,True


## 4. Mostrar las consultas al profesor

Vuelve a Workbench y ejecuta este recuento.

```sql
SELECT COUNT(*) AS total_pedidos
FROM olist.orders;
```

Después abre `sql/queries.sql`, selecciona la base `olist` y ejecuta la pregunta 3, la 4 o la 5 completa. Para el trabajo de tu compañero, abre `sql/queries_yassine.sql`.

Referencias técnicas del cargador: [pandas.to_sql](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html) y [SQLAlchemy para MySQL](https://docs.sqlalchemy.org/en/20/dialects/mysql.html#insert-on-duplicate-key-update-upsert).